<a href="https://colab.research.google.com/github/degoobd/PARDO-POLO-SISTEMAS/blob/main/Sesion12titancod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

train = pd.read_csv('/mnt/user-data/uploads/train.csv')
test = pd.read_csv('/mnt/user-data/uploads/test.csv')
full = pd.concat([train.drop(columns='Survived'), test], ignore_index=True)

def featurize(df, ref):
    d = pd.DataFrame(index=df.index)
    d['Pclass'] = df['Pclass']
    d['Sex'] = (df['Sex'] == 'male').astype(int)
    d['SibSp'] = df['SibSp']
    d['Parch'] = df['Parch']
    fam = df['SibSp'] + df['Parch'] + 1
    d['FamilySize'] = fam
    d['IsAlone'] = (fam == 1).astype(int)

    # Title extraído del nombre
    title = df['Name'].str.extract(r',\s*([^\.]+)\.')[0].str.strip()
    title = title.replace(['Mlle', 'Ms'], 'Miss').replace('Mme', 'Mrs')
    rare = ~title.isin(['Mr', 'Mrs', 'Miss', 'Master'])
    title = title.where(~rare, 'Rare')
    for t in ['Mr', 'Mrs', 'Miss', 'Master', 'Rare']:
        d['T_' + t] = (title == t).astype(int)

    # Edad: imputada por mediana de (Título, Pclass)
    age = df['Age'].copy()
    ref_title = ref['Name'].str.extract(r',\s*([^\.]+)\.')[0].str.strip()
    ref_title = ref_title.replace(['Mlle', 'Ms'], 'Miss').replace('Mme', 'Mrs')
    med = ref.assign(_t=ref_title).groupby(['_t', 'Pclass'])['Age'].median()
    for i in age[age.isna()].index:
        key = (title.loc[i], df['Pclass'].loc[i])
        d.loc[i, '_age_fill'] = med.get(key, ref['Age'].median())
    d['Age'] = age.fillna(d.get('_age_fill', pd.Series(index=df.index, dtype=float)))
    d['Age'] = d['Age'].fillna(ref['Age'].median())
    d.drop(columns=[c for c in ['_age_fill'] if c in d], inplace=True)
    d['IsChild'] = (d['Age'] < 14).astype(int)

    # Fare: por pasajero real (los tickets compartidos repiten la tarifa total)
    tcount = ref['Ticket'].value_counts()
    fmed = ref.groupby('Pclass')['Fare'].median()
    fare = df['Fare'].fillna(pd.Series(df['Pclass'].map(fmed).values, index=df.index))
    d['FarePer'] = fare / df['Ticket'].map(tcount).fillna(1)
    d['LogFare'] = np.log1p(d['FarePer'])

    d['HasCabin'] = df['Cabin'].notna().astype(int)
    deck = df['Cabin'].str[0].fillna('U')
    for k in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'U']:
        d['Deck_' + k] = (deck == k).astype(int)

    emb = df['Embarked'].fillna('S')
    for k in ['S', 'C', 'Q']:
        d['Emb_' + k] = (emb == k).astype(int)
    return d

X = featurize(train, full)
X_test = featurize(test, full)
X_test = X_test[X.columns]
y = train['Survived']

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models = {
    'Baseline (sexo)': None,
    'Regresión logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
    'Random Forest': RandomForestClassifier(n_estimators=800, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.9, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=15, random_state=42),
}
results = {}
results['Baseline (sexo)'] = (y == (1 - X['Sex'])).mean()
for name, m in models.items():
    if m is None:
        continue
    s = cross_val_score(m, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = s.mean()
    print(f'{name:24s} {s.mean():.4f} +/- {s.std():.4f}')
print('Baseline (sexo)          %.4f (en train)' % results['Baseline (sexo)'])

# Ajuste fino del mejor tipo de modelo
grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [200, 400], 'learning_rate': [0.03, 0.05, 0.1],
     'max_depth': [2, 3], 'subsample': [0.8, 1.0], 'min_samples_leaf': [1, 5]},
    cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X, y)
print('Mejor GB:', grid.best_params_, round(grid.best_score_, 4))

best = grid.best_estimator_
best.fit(X, y)
pred = best.predict(X_test).astype(int)
sub = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': pred})
sub.to_csv('/mnt/user-data/outputs/submission.csv', index=False)
print(sub.shape, sub['Survived'].mean().round(3))

imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(12).round(4).to_string())